# Introduction to Tensorflow and Keras

#### TensorFlow
- Open-Sources library for numerical computation and machine learning
- Provides tools for building and training deep learning models


#### Keras
- High-level API with TensorFlow that simplifies the process of creating and training neural networks
- Key Features of Keras
  - User Friendly: Intutive Syntax for rapid prototyping
  - Modular: Building blocks for defining layers, optimizers and loss function
  - Integration: Compatible with TensorFlow for scalable deep learning tasks

## Defining Layers, Models, Compiling Networks in Keras

- Layers are building blocks of neural networks. Common types
  - **Dense (Fully Connected) Layers**
    - Each neuron is connected to every neuron in the previous layer
  - **Dropout layers**
    - Randomly drops connections to prevent overfitting
  - **Activation Layers**
    - Apply activation function to introduce non-linearity

## Building a Model
- Keras support two primary ways to define models
  - **Sequential API:** A linear stack of layers
  - **Functional API:** More flexible, allows for complex architectures

- ### Compiling a Model
  - Specifies
    - **Optimizer:** Algorithm to update Weights
    - **Loss Function:** Metric to minimize during training
    - **Metrics:** Additional performance measures
  
- ### Training, Evaluating and Saving a Model
  - Training 
    - Fit the model to data using `model.fit()`
  - Evaluation 
    - Test the model on unseen data using `model.evaluate()`
  - Saving and Loading
    - Save a trained model using `model.save()` and reload it with `keras.model.load_model`

---

### Hands-On Exercise
- Objective
  - Build, Train, evaluate, save a simple neural network to classify digits from MNIST dataset
  


In [47]:
# importing the necessary lib's
import tensorflow as tf

import numpy as np

# Loading the dataset
from tensorflow.keras.datasets import mnist

# for one hot encoding [1,0,0,0,0,0,0,]
from tensorflow.keras.utils import to_categorical

# contain tools for building neural network model
# sequential is linear stack of layer, allows to add one layer at a time  going from input to output sequentially
from tensorflow.keras.models import Sequential


# Provides building blocks for layers in the neural network
# Dense: each neuron to connected other neuron
# flatten: Something that flattens the multi-dementional input into a single vector for FCNN
# Conv2D: 2D Convlutional layer, applies filter to the input image to extract features
# MaxPooling2D: it reduces the spatial dimention of the input, while retaining the inportant features,so its used for down-sampling
# Drop-out: A regularisation technique to randomly the set the fraction of the input units to 0 during training, preventing overfitting
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout


In [61]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 0


In [62]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set memory growth to avoid allocating all GPU memory at once
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is available and configured.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training will use CPU.")

No GPU found. Training will use CPU.


In [48]:
# Load MNIST dataset
(X_train, y_train),(X_test, y_test) = mnist.load_data()


here -1,28,28,1 is the size of the images


`X_train.reshape(-1, 28, 28, 1)`
- X_train: This is the variable holding the training dataset. In the context of a dataset like MNIST (a common one for this type of code), X_train would contain a set of grayscale images of handwritten digits.

- `.reshape(...)`: This is a method from a numerical library like NumPy that changes the dimensions (or shape) of an array without changing its data.

- `(-1, 28, 28, 1)`: This is the new shape you are specifying. Let's analyze the dimensions:

- `-1`: This is a placeholder. NumPy automatically calculates this dimension based on the original size of the array and the other specified dimensions. In this case, it represents the number of samples (images) in the dataset.

- `28, 28`: These are the height and width of each image. This is a common size for datasets like MNIST.

- `1`: This represents the number of color channels. A value of 1 indicates a grayscale image. If the images were in color (e.g., RGB), this would be 3.

In [49]:
# Normalize data
X_train = X_train.reshape(-1,28,28,1).astype('float32')/255.0
X_test = X_test.reshape(-1,28,28,1).astype('float32')/255.0

In [50]:
unique_labels = np.unique(y_train)
print(unique_labels)

[0 1 2 3 4 5 6 7 8 9]


The MNIST Dataset
 - `train-images`: contains images
 - `train-labels`: contains labels of every image

There are 10 unique classes or labels inthe dataset, and its always we'll defined in the dataset

In [51]:
# One-Hot Encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [52]:
print(f"Training Data Shape:{X_train.shape}")
print(f"Test Data Shape:{X_test.shape}")


Training Data Shape:(60000, 28, 28, 1)
Test Data Shape:(10000, 28, 28, 1)


- `Sequential`: Cause we are building the sequential model
- `Conv2D`: 
  - **32**- no of filters/feature detectors
  - **(3,3)**; indicates the filter kernal size, indicates each filter is a 3x3 matrix
  - **"relu"**: sets negative values to 0, for non-linearity
  - **input_shape = (28,28,1)** :  specifies the  shape of the input image
    - `(28,28)` = size of the input image
    - `1`: it is a channel indicates the grayscale
- `MaxPooling2D(2,2)`: adds the maxppooling2D layer with 2x2 poolsize meaning it reduces each 2x2 block in the input into single maximum value, this reduces the spatial dimentions by half
- `Flatten()`: adds the flatten layer, to convert the 2D feature map output of the previous layer into a 1D vector, necessary for connecting Convilutional layers to fully connected layers
- `Dense(128, activation="relu")`: Adding the dense layer with 128 neuron and applys the relu activation function
- `Dropout(0.5):` Adds a dropout layer, 0.5 is a fraction of neurons, to randomly drop during training, **helps prevent overfitting**
- `Dense(10,activation="softmax")`: output layer for the classification
  - `10` neurons corresponds to 10 ouput classes eg (0  to 9)
  - Activation Softmax: outputs the probability distribution across those 10 different classes

In [53]:
# Build the model
model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape = (28,28,1)),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10,activation="softmax")
])

In [54]:
# Display model Architechture
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape      ┃    Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)        │ (None, 26, 26,    │        320 │
│                          │ 32)               │            │
├──────────────────────────┼───────────────────┼────────────┤
│ max_pooling2d_3          │ (None, 13, 13,    │          0 │
│ (MaxPooling2D)           │ 32)               │            │
├──────────────────────────┼───────────────────┼────────────┤
│ flatten_3 (Flatten)      │ (None, 5408)      │          0 │
├──────────────────────────┼───────────────────┼────────────┤
│ dense_6 (Dense)          │ (None, 128)       │    692,352 │
├──────────────────────────┼───────────────────┼────────────┤
│ dropout_3 (Dropout)      │ (None, 128)       │          0 │
├──────────────────────────┼───────────────────┼────────────┤
│ dense_7 (Dense)          │ (None, 10)        │      1,290 │
└──────────────────────────┴───────────────────┴────────────┘

 Total params: 693,962 (2.65 MB)

 Trainable params: 693,962 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
# Compile the model
model.compile(
    optimizer= "adam",
    loss="categorical_crossentropy",
    metrics = ['accuracy']
)

In [56]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs = 10,
    batch_size = 32,
    validation_split = 0.2
    )

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 34s 21ms/step - accuracy: 0.9088 - loss: 0.3068 - val_accuracy: 0.9736 - val_loss: 0.0929
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 30s 20ms/step - accuracy: 0.9619 - loss: 0.1283 - val_accuracy: 0.9786 - val_loss: 0.0716
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - accuracy: 0.9700 - loss: 0.0973 - val_accuracy: 0.9827 - val_loss: 0.0591
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.9775 - loss: 0.0758 - val_accuracy: 0.9845 - val_loss: 0.0524
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.9797 - loss: 0.0648 - val_accuracy: 0.9856 - val_loss: 0.0515
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 27s 18ms/step - accuracy: 0.9822 - loss: 0.0565 - val_accuracy: 0.9861 - val_loss: 0.0508
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - accuracy: 0.9840 - loss: 0.0500 - val_accuracy: 0.9859 - val_loss: 0.0505
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 29s 20ms/step - accuracy: 0.9855 -

In [57]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test,y_test)
print(f"Test Accuracy:{test_accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9865 - loss: 0.0471
Test Accuracy:0.9865


In [58]:
# Save the model
model.save('mnist_classifier.h5')

In [59]:
# Load the model
from tensorflow.keras.models import load_model
loaded_model = load_model('mnist_classifier.h5')

:.4f: This is the format specifier. It controls how the accuracy value is displayed.

:: Separates the variable name from the format specifier.

.4: Specifies that the number should be rounded to four decimal places.

f: Indicates that the number should be treated as a floating-point number (a number with a decimal point).

In [60]:
# Verify loadede model performance
loss, accuracy = loaded_model.evaluate(X_test, y_test)
print(f"Loaded Model Accuracy:{accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9865 - loss: 0.0471
Loaded Model Accuracy:0.9865
